In [ ]:
from io import StringIO
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, Markdown

%matplotlib inline

SAVE_FIGURES = False
SHOW_CONFUSION_MATRICES = True
PROJECT_ROOT = next((parent for parent in (Path.cwd(), *Path.cwd().parents)
                     if (parent / "src/scripts/generate_paper_figures.py").is_file()), Path.cwd())
FIGURE_DIR = PROJECT_ROOT / "figures" / "notebook"

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.titlesize": 13,
    "axes.labelsize": 11, "axes.spines.top": False, "axes.spines.right": False,
    "savefig.facecolor": "white",
})

def finish(fig, filename):
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURE_DIR / f"{filename}.png", dpi=180, bbox_inches="tight")
        fig.savefig(FIGURE_DIR / f"{filename}.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 2. Embedded data

Rows come from:
- `reports/modeling/complete_cases/metrics.csv`
- `reports/modeling/ensemble_v1/voting/metrics.csv`
- `reports/modeling/deep_v1/{mlp,resnet,tabpfn}/metrics.csv`
- `reports/modeling/hybrid_attention_v1/metrics.csv`

Hybrid paired intervals compare against original fixed-C Logistic Regression, consistent with this plot's baseline. Tuned Logistic Regression is shown as a separate model; no unrecorded interval is assigned to it.

Paired intervals come from each corresponding `paired_vs_logistic.csv`. Repeated classical reference rows agree and are shown once. Accuracy is derived from (TN + TP) / N. Other metrics preserve the source values.

Only aggregate metrics are embedded. ROC/PR curves and probability-calibration curves cannot be reconstructed from these summaries, so the notebook does not invent them.


In [ ]:
METRICS_CSV = """experiment,model,source,participants,accuracy,precision,sensitivity,specificity,f1,balanced_accuracy,auroc,average_precision,brier_score,tn,fp,fn,tp
clinical_reference,logistic_regression,original,589,0.8438030560271647,0.81,0.75,0.8981233243967829,0.7788461538461539,0.8240616621983914,0.9120494489127198,0.8520045910979951,0.10969188468405548,335,38,54,162
clinical_reference,tuned_logistic_regression,hybrid_attention,589,0.8522920203735145,0.8208955223880597,0.7638888888888888,0.903485254691689,0.7913669064748201,0.8336870717902889,0.9122852745506901,0.8529996555910079,0.11097836547162153,337,36,51,165
clinical_reference,random_forest,original,589,0.8404074702886248,0.805,0.7453703703703703,0.8954423592493298,0.7740384615384616,0.8204063648098501,0.9053222122927216,0.8487166679852318,0.11989597246870903,334,39,55,161
clinical_reference,xgboost,original,589,0.8268251273344652,0.7821782178217822,0.7314814814814815,0.8820375335120644,0.7559808612440191,0.8067595074967729,0.9027281302750471,0.8434914649208308,0.11953276102014038,329,44,58,158
clinical_reference,extra_trees,voting,589,0.833616298811545,0.8041237113402062,0.7222222222222222,0.8981233243967829,0.7609756097560976,0.8101727733095025,0.9034356071889584,0.845087657102278,0.12414166220222439,335,38,60,156
clinical_reference,equal_voting,voting,589,0.8370118845500849,0.7970297029702971,0.7453703703703703,0.8900804289544236,0.7703349282296651,0.817725399662397,0.9131541058484759,0.8616796751609428,0.11278853879628045,332,41,55,161
clinical_reference,weighted_voting,voting,589,0.8455008488964346,0.8078817733990148,0.7592592592592593,0.8954423592493298,0.7828162291169452,0.8273508092542945,0.9122728626750075,0.8651478084700023,0.11181570231549325,334,39,52,164
clinical_reference,mlp_seed42,mlp,589,0.8387096774193549,0.8102564102564103,0.7314814814814815,0.900804289544236,0.7688564476885644,0.8161428855128587,0.9097036044086982,0.8515932533128315,0.11371036236212799,336,37,58,158
clinical_reference,mlp_seed43,mlp,589,0.8455008488964346,0.8238341968911918,0.7361111111111112,0.9088471849865952,0.7775061124694377,0.8224791480488531,0.9099518419223513,0.85144090459315,0.11254721710589512,339,34,57,159
clinical_reference,mlp_seed44,mlp,589,0.8455008488964346,0.8109452736318408,0.7546296296296297,0.8981233243967829,0.7817745803357314,0.8263764770132063,0.9110192632310594,0.8508243056016669,0.11184815946843649,335,38,53,163
clinical_reference,mlp_seed_mean,mlp,589,0.8404074702886248,0.8144329896907216,0.7314814814814815,0.903485254691689,0.7707317073170732,0.8174833680865853,0.9110068513553768,0.8524196901283662,0.1120582167636567,337,36,58,158
clinical_reference,resnet_seed42,resnet,589,0.8285229202037352,0.7832512315270936,0.7361111111111112,0.8820375335120644,0.7589498806682577,0.8090743223115877,0.8946231754542746,0.8365089070266006,0.12309629569479001,329,44,57,159
clinical_reference,resnet_seed43,resnet,589,0.8404074702886248,0.805,0.7453703703703703,0.8954423592493298,0.7740384615384616,0.8204063648098501,0.89464799920564,0.8547701153448741,0.11832350491170329,334,39,55,161
clinical_reference,resnet_seed44,resnet,589,0.8539898132427843,0.8186274509803921,0.7731481481481481,0.900804289544236,0.7952380952380952,0.836976218846192,0.8986942706781847,0.8411118385524288,0.11398143895638224,336,37,49,167
clinical_reference,resnet_seed_mean,resnet,589,0.8387096774193549,0.8009950248756219,0.7453703703703703,0.8927613941018767,0.7721822541966427,0.8190658822361235,0.9034604309403237,0.8565897521807394,0.11511200114106458,333,40,55,161
clinical_reference,tabpfn_seed42,tabpfn,589,0.8455008488964346,0.7990430622009569,0.7731481481481481,0.8873994638069705,0.7858823529411765,0.8302738059775594,0.916691490418032,0.8629192851273173,0.10932828228684771,331,42,49,167
clinical_reference,tabpfn_seed43,tabpfn,589,0.8488964346349746,0.7981220657276995,0.7870370370370371,0.8847184986595175,0.7925407925407926,0.8358777678482773,0.9164929004071095,0.8627797299283178,0.10941722056845278,330,43,46,170
clinical_reference,tabpfn_seed44,tabpfn,589,0.8488964346349746,0.8009478672985783,0.7824074074074074,0.8873994638069705,0.7915690866510539,0.8349034356071889,0.9164308410286963,0.8627818991960925,0.1094641078596046,331,42,47,169
clinical_reference,tabpfn_seed_mean,tabpfn,589,0.8471986417657046,0.8,0.7777777777777778,0.8873994638069705,0.7887323943661971,0.8325886207923742,0.91672872604508,0.8628907411898258,0.10933683301186513,331,42,48,168
clinical_reference,hybrid_seed42,hybrid_attention,589,0.8488964346349746,0.8190954773869347,0.7546296296296297,0.903485254691689,0.7855421686746988,0.8290574421606594,0.909207129381392,0.8484849427563427,0.11088986698433534,337,36,53,163
clinical_reference,hybrid_seed43,hybrid_attention,589,0.8387096774193549,0.8102564102564103,0.7314814814814815,0.900804289544236,0.7688564476885644,0.8161428855128587,0.9088720087379605,0.8484879340403755,0.11191498946974493,336,37,58,158
clinical_reference,hybrid_seed44,hybrid_attention,589,0.8488964346349746,0.8256410256410256,0.7453703703703703,0.9088471849865952,0.7834549878345499,0.8271087776784828,0.9106096713335319,0.8503941352679627,0.10984725603442366,339,34,55,161
clinical_reference,hybrid_seed_mean,hybrid_attention,589,0.8455008488964346,0.817258883248731,0.7453703703703703,0.903485254691689,0.7796610169491526,0.8244278125310297,0.910063548803495,0.8500559396649928,0.110225405649197,337,36,55,161
matched_clinical,logistic_regression,original,222,0.8063063063063063,0.8151260504201681,0.8220338983050848,0.7884615384615384,0.8185654008438819,0.8052477183833116,0.8982235984354628,0.9172078750591357,0.1300054045959498,82,22,21,97
matched_clinical,tuned_logistic_regression,hybrid_attention,222,0.8108108108108109,0.8114754098360656,0.8389830508474576,0.7788461538461539,0.825,0.8089146023468057,0.8963494132985659,0.9140424062459946,0.13225715215430908,81,23,19,99
matched_clinical,random_forest,original,222,0.7927927927927928,0.7857142857142857,0.8389830508474576,0.7403846153846154,0.8114754098360656,0.7896838331160365,0.8682366362451108,0.8785642966726016,0.1509130201227746,77,27,19,99
matched_clinical,xgboost,original,222,0.7927927927927928,0.8,0.8135593220338984,0.7692307692307693,0.8067226890756303,0.7913950456323338,0.870029335071708,0.881529121658808,0.1478766067532086,80,24,22,96
matched_clinical,extra_trees,voting,222,0.7747747747747747,0.7698412698412699,0.8220338983050848,0.7211538461538461,0.7950819672131147,0.7715938722294655,0.8639993481095176,0.8738315726069648,0.15704756818957533,75,29,21,97
matched_clinical,equal_voting,voting,222,0.7972972972972973,0.7967479674796748,0.8305084745762712,0.7596153846153846,0.8132780082987552,0.7950619295958279,0.8883637548891786,0.9027624676271766,0.1367597973602999,79,25,20,98
matched_clinical,weighted_voting,voting,222,0.8063063063063063,0.8151260504201681,0.8220338983050848,0.7884615384615384,0.8185654008438819,0.8052477183833116,0.8945567144719688,0.9117695390926952,0.13298305220159254,82,22,21,97
matched_clinical,mlp_seed42,mlp,222,0.8018018018018018,0.8135593220338984,0.8135593220338984,0.7884615384615384,0.8135593220338984,0.8010104302477183,0.8836375488917861,0.8968246207009797,0.14226227095741825,82,22,22,96
matched_clinical,mlp_seed43,mlp,222,0.8018018018018018,0.8032786885245902,0.8305084745762712,0.7692307692307693,0.8166666666666667,0.7998696219035202,0.8864080834419819,0.9037315329294024,0.1379990664215726,80,24,20,98
matched_clinical,mlp_seed44,mlp,222,0.8198198198198198,0.8305084745762712,0.8305084745762712,0.8076923076923077,0.8305084745762712,0.8191003911342895,0.8985495436766623,0.916353503960095,0.12955009302541748,84,20,20,98
matched_clinical,mlp_seed_mean,mlp,222,0.8063063063063063,0.8099173553719008,0.8305084745762712,0.7788461538461539,0.8200836820083682,0.8046773142112125,0.8938233376792698,0.9099569618051339,0.13455161591006623,81,23,20,98
matched_clinical,resnet_seed42,resnet,222,0.7927927927927928,0.8050847457627118,0.8050847457627118,0.7788461538461539,0.8050847457627118,0.7919654498044328,0.8593546284224252,0.8608049165680458,0.1510507934612563,81,23,23,95
matched_clinical,resnet_seed43,resnet,222,0.8288288288288288,0.8333333333333334,0.847457627118644,0.8076923076923077,0.8403361344537815,0.8275749674054759,0.8876303780964798,0.9084963635307266,0.13356247486326328,84,20,18,100
matched_clinical,resnet_seed44,resnet,222,0.8018018018018018,0.8083333333333333,0.8220338983050848,0.7788461538461539,0.8151260504201681,0.8004400260756193,0.8680736636245111,0.8698319878010815,0.14518514703655072,81,23,21,97
matched_clinical,resnet_seed_mean,resnet,222,0.8018018018018018,0.8083333333333333,0.8220338983050848,0.7788461538461539,0.8151260504201681,0.8004400260756193,0.8856747066492829,0.8898964836207308,0.13587734453583264,81,23,21,97
matched_clinical,tabpfn_seed42,tabpfn,222,0.7882882882882883,0.7983193277310925,0.8050847457627118,0.7692307692307693,0.8016877637130801,0.7871577574967406,0.8967568448500652,0.9170625588379108,0.13271990350354362,80,24,23,95
matched_clinical,tabpfn_seed43,tabpfn,222,0.7882882882882883,0.7933884297520661,0.8135593220338984,0.7596153846153846,0.803347280334728,0.7865873533246415,0.8956160365058671,0.9151364607110761,0.13286709682744122,79,25,22,96
matched_clinical,tabpfn_seed44,tabpfn,222,0.7837837837837838,0.7916666666666666,0.8050847457627118,0.7596153846153846,0.7983193277310925,0.7823500651890483,0.8938233376792699,0.9128898409348196,0.13367889010016973,79,25,23,95
matched_clinical,tabpfn_seed_mean,tabpfn,222,0.7882882882882883,0.7983193277310925,0.8050847457627118,0.7692307692307693,0.8016877637130801,0.7871577574967406,0.8952086049543676,0.9148279955272529,0.13293781462583296,80,24,23,95
matched_clinical,hybrid_seed42,hybrid_attention,222,0.7927927927927928,0.8103448275862069,0.7966101694915254,0.7884615384615384,0.8034188034188035,0.7925358539765319,0.8806225554106911,0.8968053462605682,0.143728171989999,82,22,24,94
matched_clinical,hybrid_seed43,hybrid_attention,222,0.8018018018018018,0.8135593220338984,0.8135593220338984,0.7884615384615384,0.8135593220338984,0.8010104302477183,0.8873859191655803,0.9057752221012625,0.139388250351495,82,22,22,96
matched_clinical,hybrid_seed44,hybrid_attention,222,0.8108108108108109,0.8114754098360656,0.8389830508474576,0.7788461538461539,0.825,0.8089146023468057,0.8961864406779662,0.911362990036757,0.13165411179587644,81,23,19,99
matched_clinical,hybrid_seed_mean,hybrid_attention,222,0.8063063063063063,0.8205128205128205,0.8135593220338984,0.7980769230769231,0.8170212765957446,0.8058181225554107,0.8941492829204695,0.9092154151116325,0.13399359258660326,83,21,22,96
matched_fusion,logistic_regression,original,222,0.8378378378378378,0.847457627118644,0.847457627118644,0.8269230769230769,0.847457627118644,0.8371903520208605,0.90620925684485,0.9191385854602319,0.12360723601390543,86,18,18,100
matched_fusion,tuned_logistic_regression,hybrid_attention,222,0.8288288288288288,0.8333333333333334,0.847457627118644,0.8076923076923077,0.8403361344537815,0.8275749674054759,0.9030312907431551,0.9138017299228609,0.1251442543180635,84,20,18,100
matched_fusion,random_forest,original,222,0.7927927927927928,0.7857142857142857,0.8389830508474576,0.7403846153846154,0.8114754098360656,0.7896838331160365,0.8741851368970014,0.8874875332404488,0.14867536079670385,77,27,19,99
matched_fusion,xgboost,original,222,0.8153153153153153,0.8290598290598291,0.8220338983050848,0.8076923076923077,0.825531914893617,0.8148631029986962,0.8877933507170795,0.8970954184550651,0.13921143760576413,84,20,21,97
matched_fusion,extra_trees,voting,222,0.8018018018018018,0.7936507936507936,0.847457627118644,0.75,0.819672131147541,0.798728813559322,0.8759778357235984,0.8843381328459115,0.1505002073431014,78,26,18,100
matched_fusion,equal_voting,voting,222,0.8243243243243243,0.8319327731092437,0.8389830508474576,0.8076923076923077,0.8354430379746836,0.8233376792698827,0.9017275097783574,0.916382604957093,0.1304557010658363,84,20,19,99
matched_fusion,weighted_voting,voting,222,0.8288288288288288,0.8389830508474576,0.8389830508474576,0.8173076923076923,0.8389830508474576,0.828145371577575,0.90620925684485,0.9214909843585984,0.1249234221202537,85,19,19,99
matched_fusion,mlp_seed42,mlp,222,0.8108108108108109,0.8114754098360656,0.8389830508474576,0.7788461538461539,0.825,0.8089146023468057,0.895045632333768,0.904684942405071,0.1304891411906453,81,23,19,99
matched_fusion,mlp_seed43,mlp,222,0.8198198198198198,0.8095238095238095,0.864406779661017,0.7692307692307693,0.8360655737704918,0.8168187744458931,0.8954530638852672,0.8903635253704979,0.1302788955498251,80,24,16,102
matched_fusion,mlp_seed44,mlp,222,0.7927927927927928,0.8,0.8135593220338984,0.7692307692307693,0.8067226890756303,0.7913950456323338,0.8946382007822685,0.9090836838000343,0.13380571740060013,80,24,22,96
matched_fusion,mlp_seed_mean,mlp,222,0.8063063063063063,0.8099173553719008,0.8305084745762712,0.7788461538461539,0.8200836820083682,0.8046773142112125,0.8964308996088657,0.9055187490912805,0.1300815243723932,81,23,20,98
matched_fusion,resnet_seed42,resnet,222,0.8243243243243243,0.8376068376068376,0.8305084745762712,0.8173076923076923,0.8340425531914893,0.8239080834419817,0.8961049543676662,0.8963338595749042,0.12976948458486132,85,19,20,98
matched_fusion,resnet_seed43,resnet,222,0.8018018018018018,0.8083333333333333,0.8220338983050848,0.7788461538461539,0.8151260504201681,0.8004400260756193,0.8827411994784877,0.8832159106676658,0.14225354129771683,81,23,21,97
matched_fusion,resnet_seed44,resnet,222,0.7882882882882883,0.784,0.8305084745762712,0.7403846153846154,0.8065843621399177,0.7854465449804433,0.8964308996088658,0.9019345958243479,0.14190378632360712,77,27,20,98
matched_fusion,resnet_seed_mean,resnet,222,0.8018018018018018,0.8032786885245902,0.8305084745762712,0.7692307692307693,0.8166666666666667,0.7998696219035202,0.9023794002607562,0.9069311106026366,0.13120433824935204,80,24,20,98
matched_fusion,tabpfn_seed42,tabpfn,222,0.8108108108108109,0.8064516129032258,0.847457627118644,0.7692307692307693,0.8264462809917356,0.8083441981747066,0.903194263363755,0.9198589879621435,0.12699640878917076,80,24,18,100
matched_fusion,tabpfn_seed43,tabpfn,222,0.8018018018018018,0.8032786885245902,0.8305084745762712,0.7692307692307693,0.8166666666666667,0.7998696219035202,0.9007496740547588,0.9186540640265788,0.12872625929691514,80,24,20,98
matched_fusion,tabpfn_seed44,tabpfn,222,0.7882882882882883,0.7933884297520661,0.8135593220338984,0.7596153846153846,0.803347280334728,0.7865873533246415,0.8991199478487614,0.91731837827809,0.12910043741702656,79,25,22,96
matched_fusion,tabpfn_seed_mean,tabpfn,222,0.7972972972972973,0.8016528925619835,0.8220338983050848,0.7692307692307693,0.8117154811715481,0.795632333767927,0.9009126466753586,0.9185320197383844,0.12810376336523333,80,24,21,97
matched_fusion,hybrid_seed42,hybrid_attention,222,0.8198198198198198,0.8305084745762712,0.8305084745762712,0.8076923076923077,0.8305084745762712,0.8191003911342895,0.904335071707953,0.9190338222387825,0.12637634921634142,84,20,20,98
matched_fusion,hybrid_seed43,hybrid_attention,222,0.8198198198198198,0.825,0.8389830508474576,0.7980769230769231,0.8319327731092437,0.8185299869621904,0.903764667535854,0.9180114339467277,0.12568076628217573,83,21,19,99
matched_fusion,hybrid_seed44,hybrid_attention,222,0.7972972972972973,0.7967479674796748,0.8305084745762712,0.7596153846153846,0.8132780082987552,0.7950619295958279,0.8943122555410691,0.9087713325813378,0.13426069438042873,79,25,20,98
matched_fusion,hybrid_seed_mean,hybrid_attention,222,0.8198198198198198,0.819672131147541,0.847457627118644,0.7884615384615384,0.8333333333333334,0.8179595827900912,0.9026238591916559,0.9175607862434865,0.12721858632972227,82,22,18,100
matched_apoe,logistic_regression,original,222,0.6306306306306306,0.6551724137931034,0.6440677966101694,0.6153846153846154,0.6495726495726496,0.6297262059973925,0.6228813559322034,0.634796131221451,0.23164447360662851,64,40,42,76
matched_apoe,random_forest,original,222,0.6306306306306306,0.6551724137931034,0.6440677966101694,0.6153846153846154,0.6495726495726496,0.6297262059973925,0.619947848761408,0.6328766553573172,0.23209826391351224,64,40,42,76
matched_apoe,xgboost,original,222,0.6306306306306306,0.6551724137931034,0.6440677966101694,0.6153846153846154,0.6495726495726496,0.6297262059973925,0.6225554106910038,0.6338434114835103,0.23204899706548224,64,40,42,76"""

PAIRED_ACCURACY_CSV = """experiment,model,difference,lower_95,upper_95,bootstrap_samples
clinical_reference,equal_voting,-0.006791171477079749,-0.02716468590831922,0.01358234295415961,1000
clinical_reference,weighted_voting,0.001697792869269965,-0.01697792869269954,0.022071307300509324,1000
matched_clinical,equal_voting,-0.009009009009009028,-0.05405405405405406,0.03153153153153154,1000
matched_clinical,weighted_voting,0,-0.036036036036036,0.036036036036036,1000
matched_fusion,equal_voting,-0.013513513513513487,-0.04954954954954949,0.022522522522522515,1000
matched_fusion,weighted_voting,-0.009009009009009028,-0.04054054054054057,0.018018018018018056,1000
clinical_reference,mlp_seed42,-0.005093378607809784,-0.02037351443123936,0.010186757215619683,1000
clinical_reference,mlp_seed_mean,-0.003395585738539819,-0.01697792869269954,0.01018675721561979,1000
matched_clinical,mlp_seed42,-0.0045045045045044585,-0.036036036036036,0.027027027027027084,1000
matched_clinical,mlp_seed_mean,0,-0.027027027027027084,0.027027027027027084,1000
matched_fusion,mlp_seed42,-0.027027027027026973,-0.06306306306306297,0.009009009009009028,1000
matched_fusion,mlp_seed_mean,-0.03153153153153154,-0.06756756756756754,0.0045045045045044585,1000
clinical_reference,resnet_seed42,-0.015280135823429464,-0.032258064516129004,0.001697792869269965,1000
clinical_reference,resnet_seed_mean,-0.005093378607809784,-0.017020373514431284,0.00679117147707986,1000
matched_clinical,resnet_seed42,-0.013513513513513487,-0.05405405405405406,0.027027027027026973,1000
matched_clinical,resnet_seed_mean,-0.0045045045045044585,-0.036036036036036,0.027027027027027084,1000
matched_fusion,resnet_seed42,-0.013513513513513487,-0.05405405405405406,0.022522522522522515,1000
matched_fusion,resnet_seed_mean,-0.036036036036036,-0.07657657657657657,0,1000
clinical_reference,tabpfn_seed42,0.001697792869269965,-0.018675721561969397,0.022071307300509324,1000
clinical_reference,tabpfn_seed_mean,0.00339558573853993,-0.016977928692699432,0.02376910016977929,1000
matched_clinical,tabpfn_seed42,-0.018018018018017945,-0.058558558558558516,0.022522522522522515,1000
matched_clinical,tabpfn_seed_mean,-0.018018018018017945,-0.058558558558558516,0.022522522522522515,1000
matched_fusion,tabpfn_seed42,-0.027027027027026973,-0.06756756756756754,0.018018018018017945,1000
matched_fusion,tabpfn_seed_mean,-0.04054054054054057,-0.08108108108108114,0.0045045045045044585,1000
clinical_reference,hybrid_seed42,0.005093378607809895,-0.008488964346349714,0.018675721561969505,1000
clinical_reference,hybrid_seed_mean,0.001697792869269965,-0.01018675721561968,0.01358234295415961,1000
matched_clinical,hybrid_seed42,-0.013513513513513487,-0.04054054054054057,0.013513513513513487,1000
matched_clinical,hybrid_seed_mean,0.0,-0.018018018018018056,0.018018018018018056,1000
matched_fusion,hybrid_seed42,-0.018018018018018056,-0.05405405405405406,0.018018018018017945,1000
matched_fusion,hybrid_seed_mean,-0.018018018018018056,-0.04504504504504503,0.00900900900900892,1000"""

results = pd.read_csv(StringIO(METRICS_CSV))
paired = pd.read_csv(StringIO(PAIRED_ACCURACY_CSV))
LABELS = {
    "logistic_regression": "Logistic Regression",
    "tuned_logistic_regression": "Tuned Logistic Regression",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "extra_trees": "Extra Trees",
    "equal_voting": "Equal voting",
    "weighted_voting": "Weighted voting",
    "mlp_seed42": "MLP · seed 42 (primary)",
    "mlp_seed43": "MLP · seed 43",
    "mlp_seed44": "MLP · seed 44",
    "mlp_seed_mean": "MLP · seed-average ensemble",
    "resnet_seed42": "ResNet · seed 42 (primary)",
    "resnet_seed43": "ResNet · seed 43",
    "resnet_seed44": "ResNet · seed 44",
    "resnet_seed_mean": "ResNet · seed-average ensemble",
    "tabpfn_seed42": "TabPFN · seed 42 (primary)",
    "tabpfn_seed43": "TabPFN · seed 43",
    "tabpfn_seed44": "TabPFN · seed 44",
    "tabpfn_seed_mean": "TabPFN · seed-average ensemble",
    "hybrid_seed42": "Hybrid attention · seed 42 (primary)",
    "hybrid_seed43": "Hybrid attention · seed 43",
    "hybrid_seed44": "Hybrid attention · seed 44",
    "hybrid_seed_mean": "Hybrid attention · seed-average ensemble"
}
COHORTS = {
    "clinical_reference": "Clinical only — broader cohort (589 participants)",
    "matched_clinical": "Clinical only — matched cohort (222 participants)",
    "matched_fusion": "Clinical + APOE — matched cohort (222 participants)",
    "matched_apoe": "APOE only — matched cohort (222 participants)"
}
MODEL_ORDER = list(LABELS)

results["label"] = results["model"].map(LABELS)
results["order"] = results["model"].map({name: i for i, name in enumerate(MODEL_ORDER)})

def cohort_rows(experiment):
    return results[results.experiment == experiment].sort_values("order").copy()

print(f"{len(results)} unique model/cohort results; {len(paired)} saved paired accuracy comparisons.")


In [ ]:
FAMILY_COLORS = {
    "Reference": "#334155", "Tree models": "#2563eb", "Voting": "#d97706",
    "MLP": "#7c3aed", "ResNet": "#db2777", "TabPFN": "#059669",
    "Hybrid attention": "#0891b2",
}
def family(name):
    if name.startswith("hybrid_"): return "Hybrid attention"
    if name.startswith("mlp_"): return "MLP"
    if name.startswith("resnet_"): return "ResNet"
    if name.startswith("tabpfn_"): return "TabPFN"
    if "voting" in name: return "Voting"
    if name in {"random_forest", "xgboost", "extra_trees"}: return "Tree models"
    return "Reference"

def colors_for(names):
    return [FAMILY_COLORS[family(name)] for name in names]

metric_columns = ["accuracy", "precision", "sensitivity", "specificity", "f1",
                  "balanced_accuracy", "auroc", "average_precision", "brier_score"]
metric_labels = ["Accuracy ↑", "Precision ↑", "Sensitivity ↑", "Specificity ↑",
                 "F1 ↑", "Balanced acc. ↑", "AUROC ↑", "AP ↑", "Brier ↓"]
formats = {name: "{:.1%}" for name in metric_columns[:6]}
formats.update({name: "{:.3f}" for name in metric_columns[6:]})


## 3. Accuracy: every model and every seed

Each bar shows the percentage of participants correctly classified. The dashed line is Logistic Regression **for that same cohort**. Axes run from 0 to 100%.

The highest individual-seed bar is descriptive, not permission to select that seed after seeing the outcome.


In [ ]:
for experiment, title in COHORTS.items():
    sub = cohort_rows(experiment)
    y = np.arange(len(sub))
    fig, ax = plt.subplots(figsize=(14, max(4, 0.45 * len(sub) + 2)))
    bars = ax.barh(y, 100 * sub.accuracy, color=colors_for(sub.model), height=0.7)
    ax.set_yticks(y, sub.label)
    ax.invert_yaxis()
    baseline = float(sub.loc[sub.model == "logistic_regression", "accuracy"].iloc[0]) * 100
    ax.axvline(baseline, color="#111827", linestyle="--", linewidth=1.2,
               label=f"Logistic Regression: {baseline:.1f}%")
    ax.axvline(90, color="#dc2626", linestyle=":", linewidth=1.2, label="90% accuracy target")
    for bar, (_, row) in zip(bars, sub.iterrows()):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f"{row.accuracy:.1%}  ({int(row.tn + row.tp)}/{int(row.participants)})",
                va="center", fontsize=9)
    ax.set(xlim=(0, 105), xticks=np.arange(0, 101, 10),
           xlabel="Correct predictions (%)", title=title)
    ax.grid(axis="x", alpha=0.18)
    ax.set_axisbelow(True)
    ax.legend(loc="lower left", fontsize=9)
    fig.tight_layout()
    finish(fig, f"accuracy_{experiment}")


## 4. All nine metrics, all models

Each cell displays its actual metric. Classification metrics are percentages; AUROC/AP/Brier use 0–1.

Color represents a 0–1 score with **higher brighter**. Brier's color is reversed using `1 − Brier` because lower Brier is better, while its printed number remains the actual Brier score. Color is a visual guide, not a combined quality score; do not average the columns or compare AP across different class prevalences.


In [ ]:
for experiment, title in COHORTS.items():
    sub = cohort_rows(experiment)
    actual = sub[metric_columns].to_numpy(dtype=float)
    color_values = actual.copy()
    color_values[:, -1] = 1 - color_values[:, -1]
    fig, ax = plt.subplots(figsize=(16, max(4, len(sub) * 0.46 + 2)))
    im = ax.imshow(color_values, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_yticks(np.arange(len(sub)), sub.label)
    ax.set_xticks(np.arange(len(metric_columns)), metric_labels, rotation=25, ha="right")
    for i in range(len(sub)):
        for j in range(len(metric_columns)):
            text = f"{actual[i, j]:.1%}" if j < 6 else f"{actual[i, j]:.3f}"
            ax.text(j, i, text, ha="center", va="center", fontsize=9,
                    color="white" if color_values[i, j] < 0.55 else "#111827")
    ax.set_title(title + "\nAll metrics — higher/brighter is favorable; Brier color reversed")
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.03,
                 label="Color only: rate or 1 − Brier (not a combined score)")
    fig.tight_layout()
    finish(fig, f"all_metrics_{experiment}")


## 5. Accuracy change versus Logistic Regression, with uncertainty

The points are percentage-point changes relative to the **same-run Logistic Regression**. Horizontal lines show the recorded conditional 95% bootstrap intervals. Right of zero favors the candidate. An interval spanning or touching zero does not establish a nonzero change.

These intervals are available for voting and the prespecified seed-42 / seed-average deep and hybrid results only. There are no invented intervals for other seeds or classical models. They condition on saved predictions, omit full retraining uncertainty, and are not multiplicity-adjusted.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 10))
main_cohorts = [name for name in COHORTS if name != "matched_apoe"]
for ax, experiment in zip(axes, main_cohorts):
    sub = paired[paired.experiment == experiment].copy()
    sub["order"] = sub.model.map({name: i for i, name in enumerate(MODEL_ORDER)})
    sub = sub.sort_values("order")
    for i, (_, row) in enumerate(sub.iterrows()):
        point, low, high = 100 * row.difference, 100 * row.lower_95, 100 * row.upper_95
        color = FAMILY_COLORS[family(row.model)]
        ax.hlines(i, low, high, color=color, linewidth=2)
        ax.plot(point, i, "o", color=color, markersize=6)
        ax.annotate(f"{point:+.2f}", (point, i), xytext=(0, 7),
                    textcoords="offset points", ha="center", fontsize=8)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_yticks(np.arange(len(sub)), sub.model.map(LABELS), fontsize=8)
    ax.invert_yaxis()
    ax.set(xlim=(-9, 5), xlabel="Accuracy difference (percentage points)",
           title=COHORTS[experiment].replace(" — ", "\n"))
    ax.grid(axis="x", alpha=0.2)
fig.suptitle("Saved paired comparisons: candidate minus Logistic Regression", fontsize=15)
fig.tight_layout(rect=(0, 0, 1, 0.95))
finish(fig, "paired_accuracy_intervals")


## 6. Seed variation

Circles show the three individual seeds; diamonds show the **probability-average ensemble**. The thin gray line spans the observed individual-seed accuracies and is not a confidence interval. The dashed line is Logistic Regression.

The x-axis is explicitly zoomed to 70–90% to make seed differences visible. Seed 42 is the prespecified primary run. The ensemble accuracy need not equal the average seed accuracy.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
markers = {42: "o", 43: "s", 44: "^"}
for ax, experiment in zip(axes, main_cohorts):
    sub = cohort_rows(experiment)
    for j, (prefix, name) in enumerate([("mlp", "MLP"), ("resnet", "ResNet"), ("tabpfn", "TabPFN"), ("hybrid", "Hybrid attention")]):
        seed_rows = sub[sub.model.isin([f"{prefix}_seed{s}" for s in [42, 43, 44]])]
        ax.hlines(j, 100 * seed_rows.accuracy.min(), 100 * seed_rows.accuracy.max(),
                  color="#94a3b8", linewidth=2)
        for offset, seed in zip([-0.18, 0, 0.18], [42, 43, 44]):
            value = 100 * float(sub.loc[sub.model == f"{prefix}_seed{seed}", "accuracy"].iloc[0])
            ax.scatter(value, j + offset, color=FAMILY_COLORS[name],
                       marker=markers[seed], s=55, label=f"Seed {seed}" if j == 0 else None)
            ax.annotate(f"{seed}: {value:.1f}%", (value, j + offset),
                        xytext=(5, 1), textcoords="offset points", fontsize=8)
        ensemble = 100 * float(sub.loc[sub.model == f"{prefix}_seed_mean", "accuracy"].iloc[0])
        ax.scatter(ensemble, j + 0.35, marker="D", color="#111827", s=35,
                   label="Seed-average ensemble" if j == 0 else None)
        ax.annotate(f"avg-prob: {ensemble:.1f}%", (ensemble, j + 0.35),
                    xytext=(5, 0), textcoords="offset points", fontsize=8)
    baseline = 100 * float(sub.loc[sub.model == "logistic_regression", "accuracy"].iloc[0])
    ax.axvline(baseline, color="#475569", linestyle="--", label="Logistic Regression")
    ax.set_yticks([0, 1, 2, 3], ["MLP", "ResNet", "TabPFN", "Hybrid attention"])
    ax.set(xlim=(70, 90), ylim=(3.7, -0.5),
           xlabel="Accuracy (%) — zoomed axis", title=COHORTS[experiment].replace(" — ", "\n"))
    ax.grid(axis="x", alpha=0.2)
axes[0].legend(loc="lower left", fontsize=8)
fig.tight_layout()
finish(fig, "seed_variation")


## 7. Confusion matrices for every model

Rows are actual outcomes; columns are predicted outcomes:
- Top left: correctly identified non-converters (TN).
- Top right: false alarms (FP).
- Bottom left: missed converters (FN).
- Bottom right: correctly identified converters (TP).

Each cohort uses a common color scale across its models. Set `SHOW_CONFUSION_MATRICES = False` in the setup cell if you want fewer figures.


In [ ]:
if SHOW_CONFUSION_MATRICES:
    for experiment, title in COHORTS.items():
        sub = cohort_rows(experiment)
        cols = 4
        nrows = int(np.ceil(len(sub) / cols))
        fig, axes = plt.subplots(nrows, cols, figsize=(18, 3.3 * nrows), squeeze=False)
        vmax = int(sub[["tn", "fp", "fn", "tp"]].to_numpy().max())
        for ax, (_, row) in zip(axes.flat, sub.iterrows()):
            matrix = np.array([[row.tn, row.fp], [row.fn, row.tp]], dtype=int)
            ax.imshow(matrix, cmap="Blues", vmin=0, vmax=vmax)
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, str(matrix[i, j]), ha="center", va="center",
                            fontsize=12, color="white" if matrix[i, j] > 0.55 * vmax else "black")
            ax.set_xticks([0, 1], ["Non-converter", "Converter"], fontsize=8)
            ax.set_yticks([0, 1], ["Non-converter", "Converter"], fontsize=8)
            ax.set(xlabel="Predicted", ylabel="Actual")
            ax.set_title(f"{row.label}\nAccuracy {row.accuracy:.1%}", fontsize=9)
        for ax in axes.flat[len(sub):]:
            ax.set_visible(False)
        fig.suptitle(title + "\nConfusion counts — shared color scale within cohort", fontsize=15)
        fig.tight_layout(rect=(0, 0, 1, 0.95))
        finish(fig, f"confusion_{experiment}")


## 8. Exact tables

Use these alongside the graphs. Precision, sensitivity, specificity, F1, and accuracy are not interchangeable. No aggregate “best overall model” score is constructed.


In [ ]:
for experiment, title in COHORTS.items():
    display(Markdown(f"**{title}**"))
    sub = cohort_rows(experiment)
    display(sub[["label", "participants"] + metric_columns].set_index("label").style.format(formats))
